# **1. AI Agent**

단순히 질문에 답하는 LLM보다 한단계 확장된 시스템
- 예시) "오늘 서울 날씨를 검색해서 우산이 필요한 지 알려줘"
    - LLM 자체의 학습 지식만으로는 오늘의 실제 날씨를 알 수 없다.
    - Agent는 웹 검색이나 날씨 API 같은 tool을 호출하고 그 결과를 다시 읽어 최종 답변을 만든다.

### 핵심 구성
1. Model: 상황을 이해하고 다음 행동을 판단
2. Tools: 검색, DB 조회, 계산, API 호출 등의 실제 행동
3. State/Memory: 지금까지의 메세지와 중간 결과 보관
4. Control FLow: 어떤 순서로 실행하고 언제 반복/종료할 지 결정

# **2. RAG**

```
일반 RAG
    질문 > 검색 > 관련 문서 > LLM > 답변
```
> 검색 단계가 미리 정해져 있는 경우가 많다.
```
Agentic RAG
    질문 > LLM 판단 > 검색 여부 > 검색 > 검색 결과 판단 > 재검색/다른 tool > 답변
```
> 검색 자체가 하나의 tool이 되고, agent가 검색 여부와 다음 행동을 판단한다.

# **3. LangGraph**

Agent는 한번의 LLM 호출로 끝나지 않을 수 있다.
- 예시) 사용자 > LLM > 검색 tool > LLM > rPtks tool > LLM > 최종 답변

    위 흐름에 다음 요구사항이 붙으면 더 복잡해진다.
    - 검색 결과가 부족하면 다시 검색
    - 특정 조건이면 사람의 승인 요청
    - 실패하면 재시도
    - 대화 상태 저장
    - 여러 노드를 병렬 실행

### LangGraph의 핵심 개념
1. State : 그래프가 실행되는 동안 여러 노드가 공유하는 데이터
2. Node: 실제 작업을 수행하는 함수
3. Edge: 한 Node 다음에 어떤 Node를 실행할지 연결
4. Conditional Edge: State를 보고 다음 경로를 동적으로 선택

In [1]:
from typing_extensions import TypedDict

In [2]:
class UserDict(TypedDict):
    user_id: int
    user_name: str
    user_email: str

user = {'id': 1, 'name': '김사과', 'email': 'apple@apple.com'}

In [3]:
from pydantic import BaseModel

In [4]:
class UserModel(BaseModel):
    id: int
    name: str
    email: str

user = UserModel(id=1, name="김사과", email="apple@apple.com")
print(user)
print(user.model_dump())

id=1 name='김사과' email='apple@apple.com'
{'id': 1, 'name': '김사과', 'email': 'apple@apple.com'}


In [5]:
from langgraph.graph import StateGraph, START, END

In [6]:
class HelloState(TypedDict):
    name: str
    greeting: str

def make_greeting(state: HelloState):
    return {"greeting": f"안녕하세요, {state['name']}님!"}

In [7]:
builder = StateGraph(HelloState)

builder.add_node("greeting", make_greeting)
builder.add_edge(START, "greeting")
builder.add_edge("greeting", END)

hello_graph = builder.compile()

In [8]:
result = hello_graph.invoke({"name": "김사과", "greeting": ""})
print(result)

{'name': '김사과', 'greeting': '안녕하세요, 김사과님!'}


In [9]:
# 노드의 기본 패턴
# 1. State 읽기
# 2. 작업하기
# 3. 업데이트할 값 반환
class CounterState(TypedDict):
    count: int

def plus_one(state: CounterState):
    return {"count": state["count"] + 1}

builder = StateGraph(CounterState)

builder.add_node("plus_one", plus_one)
builder.add_edge(START, "plus_one")
builder.add_edge("plus_one", END)

counter_graph = builder.compile()
counter_graph.invoke({"count": 10}) # update 되서 반환

{'count': 11}

### Reducer
여러 노드가 같은 State 필드를 업데이트할 때 기존 값을 덮어 쓸지, 합칠 지 정해야 한다.
- 기본값은 대체(overwrite)

In [10]:
from typing import Annotated
from operator import add

In [11]:
# Annotated[list[str], add]: 새 노드를 반환했을 때, 기존 리스트를 버리는 것이 아니라 기존 리스트에 새 노드를 추가
class ListState(TypedDict):
    history: Annotated[list[str], add]

def first(state: ListState):
    return {"history": ["A"]}

def second(state: ListState):
    return {"history": ["B"]}

In [12]:
builder = StateGraph(ListState)

builder.add_node("first", first)
builder.add_node("second", second)
builder.add_edge(START, "first")
builder.add_edge("first", "second")
builder.add_edge("second", END)

list_graph = builder.compile()
list_graph.invoke({"history": []})

{'history': ['A', 'B']}

In [13]:
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage
from langgraph.graph.message import add_messages

In [14]:
class MessageState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages] # LLM 메세지 누적

def simple_chatbot(state: MessageState):
    return {
        "messages": [AIMessage(content="안녕하세요! 무엇을 도와드릴까요?")] # HumanMessage에 대한 답변
    }

builder = StateGraph(MessageState)

builder.add_node("chatbot", simple_chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

message_graph = builder.compile()

result = message_graph.invoke({
    "messages": [HumanMessage(content="안녕")] # 사용자의 입력
})

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

안녕
================================== Ai Message ==================================

안녕하세요! 무엇을 도와드릴까요?


In [15]:
class PipelineState(TypedDict):
    text: str
    length: int

def clean_text(state: PipelineState):
    return {"text": state["text"].strip()}

def to_upper(state: PipelineState):
    return {"text": state["text"].upper()}

def count_length(state: PipelineState):
    return {"length": len(state["text"])}

builder = StateGraph(PipelineState)

builder.add_node("clean", clean_text)
builder.add_node("upper", to_upper)
builder.add_node("count", count_length)

builder.add_edge(START, "clean")
builder.add_edge("clean", "upper")
builder.add_edge("upper", "count")
builder.add_edge("count", END)

pipeline_graph = builder.compile()
pipeline_graph.invoke({
    "text": "   hello agent   ",
    "length": 0
})

{'text': 'HELLO AGENT', 'length': 11}

In [16]:
from typing import Literal

In [19]:
class RouteState(TypedDict):
    score: int
    result: str

def check_score(state: RouteState):
    return {}

def pass_node(state: RouteState):
    return {"result": "통과"}

def fail_node(state: RouteState):
    return {"result": "재학습 필요"}

def route_score(state: RouteState) -> Literal["pass_node", "fail_node"]:
    if state["score"] >= 60:
        return "pass_node"
    else:
        return "fail_node"

In [20]:
builder = StateGraph(RouteState)
builder.add_node("check", check_score)
builder.add_node("pass_node", pass_node)
builder.add_node("fail_node", fail_node)

builder.add_edge(START, "check")
builder.add_conditional_edges("check", route_score)
builder.add_edge("pass_node", END)
builder.add_edge("fail_node", END)

route_graph = builder.compile()

In [21]:
print(route_graph.invoke({"score": 85, "result": ""}))
print(route_graph.invoke({"score": 45, "result": ""}))

{'score': 85, 'result': '통과'}
{'score': 45, 'result': '재학습 필요'}


In [25]:
class RetryState(TypedDict):
    count: int

def increase(state: RetryState):
    print("현재 count:", state["count"])
    return {"count": state["count"] + 1}

def continue_or_end(state: RetryState) -> Literal["increase", "__end__"]:
    if state["count"] < 20:
        return "increase"
    return END

builder = StateGraph(RetryState)
builder.add_node("increase", increase)
builder.add_edge(START, "increase")
builder.add_conditional_edges("increase", continue_or_end)

retry_graph = builder.compile()

retry_graph.invoke({"count": 0})

현재 count: 0
현재 count: 1
현재 count: 2
현재 count: 3
현재 count: 4
현재 count: 5
현재 count: 6
현재 count: 7
현재 count: 8
현재 count: 9
현재 count: 10
현재 count: 11
현재 count: 12
현재 count: 13
현재 count: 14
현재 count: 15
현재 count: 16
현재 count: 17
현재 count: 18
현재 count: 19


{'count': 20}

In [28]:
from langgraph.errors import GraphRecursionError

In [30]:
try:
    result = retry_graph.invoke(
        {"count": 0},
        config={"recursion_limit": 10}
    )
    print(result)
except GraphRecursionError:
    print("그래프 반복 횟수 제한을 초과했습니다.")

현재 count: 0
현재 count: 1
현재 count: 2
현재 count: 3
현재 count: 4
현재 count: 5
현재 count: 6
현재 count: 7
현재 count: 8
현재 count: 9
그래프 반복 횟수 제한을 초과했습니다.


### 병렬 Branch

In [31]:
class ParallelState(TypedDict):
    results: Annotated[list[str], add]

def start_node(state: ParallelState):
    return {"results": ["START"]}

def branch_a(state: ParallelState):
    return {"results": ["A"]}

def branch_b(state: ParallelState):
    return {"results": ["B"]}

def finish_node(state: ParallelState):
    return {"results": ["FINISH"]}

builder = StateGraph(ParallelState)
builder.add_node("start_node", start_node)
builder.add_node("branch_a", branch_a)
builder.add_node("branch_b", branch_b)
builder.add_node("finish", finish_node)

builder.add_edge(START, "start_node")
builder.add_edge("start_node", "branch_a")
builder.add_edge("start_node", "branch_b")
builder.add_edge("branch_a", "finish")
builder.add_edge("branch_b", "finish")
builder.add_edge("finish", END)

parallel_graph = builder.compile()
parallel_graph.invoke({"results": []})

{'results': ['START', 'A', 'B', 'FINISH']}

### 실행 방식

In [32]:
result = message_graph.invoke({
    "messages": [{"role": "user", "content": "안녕하세요"}]
})
print(result)

{'messages': [HumanMessage(content='안녕하세요', additional_kwargs={}, response_metadata={}, id='0e395376-3c51-415c-b37f-716bc0c0021a'), AIMessage(content='안녕하세요! 무엇을 도와드릴까요?', additional_kwargs={}, response_metadata={}, id='7ddf4bd2-af21-4faf-96b5-44e4cbb5613b', tool_calls=[], invalid_tool_calls=[])]}


In [33]:
for chunk in message_graph.stream(
    {"messages": [{"role": "user", "content": "안녕하세요"}]},
    stream_mode="updates"
):
    print(chunk)

{'chatbot': {'messages': [AIMessage(content='안녕하세요! 무엇을 도와드릴까요?', additional_kwargs={}, response_metadata={}, id='c08a8131-15d6-4bc5-af76-4baa8337d720', tool_calls=[], invalid_tool_calls=[])]}}


In [34]:
result = await message_graph.ainvoke({
    "messages": [{"role": "user", "content": "비동기 호출 테스트"}]
})
print(result)

{'messages': [HumanMessage(content='비동기 호출 테스트', additional_kwargs={}, response_metadata={}, id='f05493bc-14c7-448c-a222-a41071bc686b'), AIMessage(content='안녕하세요! 무엇을 도와드릴까요?', additional_kwargs={}, response_metadata={}, id='6ec75c2e-a10e-4206-8c8d-6183fe236fa0', tool_calls=[], invalid_tool_calls=[])]}


# **4. LLM과 Tool calling**

Tool Calling에서 LLM은 Python 함수를 직접 실행하는 것이 아니라 LLM은 도구 호출 요청을 생성하는 것이다.
```
{
    "name": "multiply",
    "args": {"a":3, "b":12}
}
```

In [36]:
from langchain_core.tools import tool

In [ ]:
@tool
def add(a: int, b: int) -> int:
    """두 정수를 더합니다""" # tool에 대한 함수의 정의나 설명
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """두 정수를 곱합니다"""
    return a * b

In [38]:
tools = [add, multiply]

print(add.invoke({"a": 2, "b": 20}))
print(multiply.invoke({"a": 3, "b": 12}))

22
36


### ChatOpenAI에 Tool 연결

In [42]:
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

True

In [44]:
MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
MODEL_NAME

'gpt-5.6-luna'

In [45]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [48]:
llm = ChatOpenAI(model_name=MODEL_NAME, openai_api_key=OPENAI_API_KEY, temperature=0, reasoning_effort="none")
llm_with_tools = llm.bind_tools(tools)

In [ ]:
response = llm_with_tools.invoke("3 곱하기 12를 계산하고, 11 더하기 49도 계산해줘.")

print("content:", response.content)
print("tool_calls:", response.tool_calls)

bind_tools()를 했다고 tool이 자동으로 실행되는 것이 아니라 LLM이 tool_calls()를 생성한 뒤, 누군가 tool을 실행해야 한다.

In [54]:
# MessagesState에는 message 필드와 add_messages reducer가 이미 준비되어 있어서 대화형 Agent에서는 매우 편리
from langgraph.graph import MessagesState
# ToolNode는 마지막 AIMessage의 tool_calls를 읽고 실제 tool을 실행한 뒤, 결과를 ToolMeaasage 형태로 state에 추가
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition
# 흐름: HumanMessage > LLM > AIMessage(tool_calls) > ToolNode > ToolMessage > LLM

In [51]:
def call_model(state: MessagesState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

In [55]:
tool_node = ToolNode(tools)

```
START > agent(LLM) > tool 필요? 아니오 > END
                                예 > tools > agent
```

In [56]:
builder = StateGraph(MessagesState)

builder.add_node("agent", call_model)
builder.add_node("tools", tool_node)

builder.add_edge(START, "agent")

builder.add_conditional_edges("agent", tools_condition)

builder.add_edge("tools", "agent")

tool_agent_graph = builder.compile()

In [ ]:
result = tool_agent_graph.invoke({
    "messages": [
        {
            "role": "user",
            "content": "37 곱하기 21을 계산해줘."
        }
    ]
})

for message in result["messages"]:
    message.pretty_print()

In [58]:
from langchain.agents import create_agent

In [59]:
simple_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "당신은 친절한 AI 수업용 에이전트입니다. "
        "계산이 필요하면 제공된 도구를 사용하세요."
    )
)

In [ ]:
result = simple_agent.invoke({
    "messages": [
        {"role": "user", "content": "123과 456을 더해줘."}
    ]
})

result["messages"][-1].pretty_print()

# **5. 웹 검색 Tool 추가**

LLM이 학습한 지식만 사용하는 것이 아니라, 필요한 경우 현재 웹 정보를 tool로 검생할 수 있도록 한다.

In [64]:
from langchain_tavily import TavilySearch

load_dotenv()

True

In [66]:
web_search = TavilySearch(max_results=3)

web_search.invoke({"query": "LangGraph 최신 기능"})

{'query': 'LangGraph 최신 기능',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.youtube.com/watch?v=Ipa6JNbFq4g',
   'title': 'LangChain/LangGraph V1 업데이트 이후 신기능 활용하여 에이전트 제작하기!',
   'content': 'LangChain과 LangGraph가 V1으로 대대적인 업데이트를 단행하며, 에이전트 개발의 패러다임이 새롭게 정립되고 있습니다. 본 세션에서는 실무에서 바로 활용 가능한 최신 기능들을 심층적으로 살펴봅니다. 먼저, LangSmith의 신기능인 Insights를 통해 에이전트의 성능을 데이터 기반으로 분석하고, Multi-turn Evaluation으로 복잡한 대화형 에이전트의 품질을 체계적으로 평가하는 방법을 다룹니다. 또한 Polly를 활용한 Tracing 디버깅으로 에이전트의 동작을 시각적으로 추적하고 문제를 빠르게 진단하는 실전 기법을 공유합니다. 다음으로, LangGraph V1의 핵심 신기능인 create_agent API를 통해 에이전트를 보다 직관적으로 구축하는 방법과, Built-in/Custom Middleware를 활용하여 인증, 로깅, 에러 처리 등 공통 관심사를 깔끔하게 분리하는 아키텍처 설계 패턴을 소개합니다. 마지막으로, 복잡한 태스크를 처리하는 [...] [12:25] 저희가 맨날 랭체인 랭체인 부르는게 이제 랭체인이 있고 그리고 이제 최근에 화두가 된 뭐 최근이 아니죠\n[12:33] 뭐 조금 조금 됐 됐지만 2년 정도 됐지만 이제 랭그래프가 있습니다.\n[12:37] 대부분은 이제 최신 트렌드가 다 랭그래프 쪽으로 넘어가고 있고 랭체인도 랭그래프에 굉장히 집중하고 있습니다. 그래서 이제 시작하시는 분들은 저는 랭그래프부터 보셔도\n[12:45] 좋다라는 말씀을 드리고 싶고요. 또 요런 것들을 모니터링하기 

In [ ]:
search_agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt=(
        "당신은 웹 검색을 사용할 수 있는 리서치 에이전트입니다. "
        "최신 정보가 필요하면 검색 도구를 사용하고, "
        "검색 결과를 바탕으로 간결하게 답하세요."
    )
)

result = search_agent.invoke({
    "messages": [
        {"role": "user", "content": "LangGraph가 무엇인지 최신 내용을 검색해서 설명해줘."}
    ]
})
result["messages"][-1].pretty_print()

# **6. Structured Output**

Agent 결과를 사람이 읽기만 한다면 문자열도 충분하기에 다른 시스템으로 값을 전달하려면 정해진 구조가 유리하다.

In [68]:
from pydantic import BaseModel, Field

In [69]:
class MovieResponse(BaseModel):
    title: str = Field(description="영화 제목")
    director: str = Field(description="감독")
    genre: str = Field(description="장르")
    release_year: int = Field(description="개봉 연도")

structured_llm = llm.with_structured_output(MovieResponse)

In [ ]:
movie = structured_llm.invoke("영화 인터스텔라 정보를 구조화해서 알려줘.")
movie

In [ ]:
print(movie.title)
print(movie.director)
print(movie.genre)
print(movie.release_year)